In [1]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from collections import OrderedDict
import re
import dgl
from torch.utils.data import DataLoader
import math 
import torch.nn.functional as F
import dgl.function as fn
from dgl.nn.functional import edge_softmax
import tqdm
import time
from sklearn.metrics import accuracy_score, f1_score, recall_score, precision_score

F:\anaconda\envs\wppenv\lib\site-packages\tqdm\auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
encode_dict = np.load('com_encode_dict256tb.npy',allow_pickle=True).item()
encode_dict['C15'].shape

(4, 256)

In [7]:
from dgl.data.utils import load_graphs
from dgl.data import DGLDataset
class TrainDataset(DGLDataset):
    def __init__(self):
        super(TrainDataset,self).__init__(name='TrainDataset')
    def process(self):
        self.graphs = []
        self.labels = []
        self.graphs, label_dict = load_graphs("Thunderbird_Train_fixseq.bin")
        self.labels = label_dict['labels']
    def __getitem__(self, idx):
        return self.graphs[idx], self.labels[idx]
    def __len__(self):
        return len(self.graphs)

In [8]:
class TestDataset(DGLDataset):
    def __init__(self):
        super(TestDataset,self).__init__(name='TestDataset')
    def process(self):
        self.graphs = []
        self.labels = []
        self.graphs, label_dict = load_graphs("Thunderbird_Test_fixseq.bin")
        self.labels = label_dict['labels']
    def __getitem__(self, idx):
        return self.graphs[idx], self.labels[idx]
    def __len__(self):
        return len(self.graphs)

In [9]:
Traindataset = TrainDataset()

In [10]:
Testdataset = TestDataset()

In [11]:
def _collate_fn(batch):
    graphs = []
    labels = []
    for data in batch:
        sum= 0 
        for x in data[0].ntypes:
            sum =sum+data[0].number_of_nodes(x)
        if sum<252:
            data[0].add_nodes(252-sum, ntype='C0')
        graphs.append(data[0])
        labels.append(data[1])
    labels = torch.tensor(labels, dtype=torch.long)
    return graphs, labels

In [12]:
Trainloader = DataLoader(Traindataset, batch_size=64, shuffle=False,collate_fn=_collate_fn)
Testloader = DataLoader(Testdataset, batch_size=64, shuffle=False,collate_fn=_collate_fn)

In [13]:
len(Trainloader),len(Testloader)

(250, 63)

In [14]:
class HGTLayer(nn.Module):
    def __init__(
        self,
        in_dim,
        out_dim,
        node_dict,
        n_heads,
        dropout=0.2,
        use_norm=False,
    ):
        super(HGTLayer, self).__init__()

        self.in_dim = in_dim
        self.out_dim = out_dim
        self.node_dict = node_dict
        self.num_types = len(node_dict)
        self.n_heads = n_heads
        self.d_k = out_dim // n_heads
        self.sqrt_dk = math.sqrt(self.d_k)
        self.att = None
        self.k_linears = nn.ModuleList().cuda()
        self.q_linears = nn.ModuleList().cuda()
        self.v_linears = nn.ModuleList().cuda()
        self.a_linears = nn.ModuleList().cuda()
        self.norms = nn.ModuleList().cuda()
        self.use_norm = use_norm

        for t in range(self.num_types):
            self.k_linears.append(nn.Linear(in_dim, out_dim).cuda())
            self.q_linears.append(nn.Linear(in_dim, out_dim).cuda())
            self.v_linears.append(nn.Linear(in_dim, out_dim).cuda())
            self.a_linears.append(nn.Linear(out_dim, out_dim).cuda())
            if use_norm:
                self.norms.append(nn.LayerNorm(out_dim).cuda())


        self.skip = nn.Parameter(torch.ones(self.num_types)).cuda()
        self.drop = nn.Dropout(dropout)



    def forward(self, G, h):
        edge_dict={}
        for etype in G.canonical_etypes:
            edge_dict[etype] = len(edge_dict)
            G.edges[etype].data["id"] = (
                torch.ones(G.number_of_edges(etype), dtype=torch.long).cuda()
                * edge_dict[etype]
            )
        num_relations = len(edge_dict)
        relation_pri = nn.Parameter(
            torch.ones(num_relations, self.n_heads)
        ).cuda()
        relation_att = nn.Parameter(
            torch.Tensor(num_relations, self.n_heads, self.d_k, self.d_k)
        ).cuda()
        relation_msg = nn.Parameter(
            torch.Tensor(num_relations, self.n_heads, self.d_k, self.d_k)
        ).cuda()
        nn.init.xavier_uniform_(relation_att)
        nn.init.xavier_uniform_(relation_msg)
        with G.local_scope():
            node_dict = self.node_dict
            for srctype, etype, dsttype in G.canonical_etypes:
                sub_graph = G[srctype, etype, dsttype]

                k_linear = self.k_linears[node_dict[srctype]]
                v_linear = self.v_linears[node_dict[srctype]]
                q_linear = self.q_linears[node_dict[dsttype]]

                k = k_linear(h[srctype]).view(-1, self.n_heads, self.d_k)
                v = v_linear(h[srctype]).view(-1, self.n_heads, self.d_k)
                q = q_linear(h[dsttype]).view(-1, self.n_heads, self.d_k)
                etype_n = (srctype, etype, dsttype)
                e_id = edge_dict[etype_n]
                relation_att_e = relation_att[e_id].to(device)
                relation_pri_e = relation_pri[e_id].to(device)
                relation_msg_e = relation_msg[e_id].to(device)

                k = torch.einsum("bij,ijk->bik", k, relation_att_e)
                v = torch.einsum("bij,ijk->bik", v, relation_msg_e)

                sub_graph.srcdata["k"] = k
                sub_graph.dstdata["q"] = q
                sub_graph.srcdata["v_%d" % e_id] = v

                sub_graph.apply_edges(fn.v_dot_u("q", "k", "t"))
                attn_score = (
                    sub_graph.edata.pop("t").sum(-1)
                    * relation_pri_e
                    / self.sqrt_dk
                )
                attn_score = edge_softmax(sub_graph, attn_score, norm_by="dst")

                sub_graph.edata["t"] = attn_score.unsqueeze(-1)
            G.multi_update_all(
                {
                    etype_n: (
                        fn.u_mul_e("v_%d" % e_id, "t", "m"),
                        fn.sum("m", "t"),
                    )
                    for etype_n, e_id in edge_dict.items()
                },
                cross_reducer="mean",
            )

            new_h = {}
            for ntype in G.ntypes:
                """
                Step 3: Target-specific Aggregation
                x = norm( W[node_type] * gelu( Agg(x) ) + x )
                """
                n_id = node_dict[ntype]
                alpha = torch.sigmoid(self.skip[n_id])
                t = G.nodes[ntype].data["t"].view(-1, self.out_dim)
                trans_out = self.drop(self.a_linears[n_id](t))
                trans_out = trans_out * alpha + h[ntype] * (1 - alpha)
                if self.use_norm:
                    new_h[ntype] = self.norms[n_id](trans_out)
                else:
                    new_h[ntype] = trans_out
            return new_h

In [15]:
class HGT(nn.Module):
    def __init__(
        self,
        node_dict,
        encode_dict,
        n_inp,
        n_hid,
        n_out,
        n_layers,
        n_heads,
        use_norm=True,
    ):
        super(HGT, self).__init__()
        self.node_dict = node_dict
        self.encode_dict = encode_dict
        self.gcs = nn.ModuleList().cuda()
        self.n_inp = n_inp
        self.n_hid = n_hid
        self.n_out = n_out
        self.n_layers = n_layers
        self.adapt_ws = nn.ModuleList().cuda()
        for t in range(len(node_dict)):
            self.adapt_ws.append(nn.Linear(n_inp, n_hid).cuda())
        for _ in range(n_layers):
            self.gcs.append(
                HGTLayer(
                    n_hid,
                    n_hid,
                    node_dict,
                    n_heads,
                    use_norm=use_norm,
                ).cuda()
            )
            
        self.out = nn.Linear(252 * n_hid, n_out).cuda()
        
    def forward(self, G):
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        G_list = []
        for g in G:
            h = {}
            #注入节点特征
            g = g.to(device) 
            for ntype in g.ntypes:
                g.nodes[ntype].data["inp"] = torch.tensor(self.encode_dict[ntype][:g.number_of_nodes(ntype)]).float().cuda()  
            for ntype in g.ntypes:
                n_id = self.node_dict[ntype]
                h[ntype] = F.gelu(self.adapt_ws[n_id](g.nodes[ntype].data["inp"]))
            for i in range(self.n_layers):
                h = self.gcs[i](g, h)
            g_list = []
            for ntype in g.ntypes:
                g_list.append(h[ntype])
            g = torch.cat(g_list,0)
            seq_len = g.shape[0]
            G_list.append(g)
        input=torch.stack(G_list,dim=0)
        return self.out(input.view(input.size(0), -1))

In [16]:
def get_n_params(model):
    pp = 0
    for p in list(model.parameters()):
        nn = 1
        for s in list(p.size()):
            nn = nn * s
        pp += nn
    return pp

In [17]:
# train_step = torch.tensor(0)

In [18]:
# train_step+1

tensor(1)

In [19]:
# for inputs,label in Trainloader:
#     for g in inputs:
#         for ntype in g.ntypes:
#             g.nodes[ntype].data["inp"] = torch.tensor(encode_dict[ntype][:g.number_of_nodes(ntype)]).float().cuda() 

KeyError: 'C0'

In [20]:
def time_since(since):
    s = time.time() - since
    m = math.floor(s/60)
    s-= m*60
    return '%dm %ds' % (m, s)

In [21]:
def train(model,train_step):
    total_loss = 0
    counter = 0
    for inputs,label in Trainloader:
        model.train()
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        logits = model(inputs)
        # The loss is computed only for labeled nodes.
        loss = F.cross_entropy(logits, label.to(device))
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), clip)
        optimizer.step()
        scheduler.step(train_step)
        #计算准确率
        pred = logits.max(dim=1, keepdim=True)[1]
#         y_pred.extend(pred.tolist())  
#         y_true.extend(label.tolist())
        counter += 1
        total_loss += loss.item()
        if counter % 10 == 0:
            print(f'[{time_since(start)}] Epoch {epoch} ', end='')
            print(f'[{counter}/{len(Trainloader)}] ', end='')
            print(f'loss={total_loss / (counter * len(inputs))}')


In [22]:
def test(model):
    counter = 0
    y_pred = []
    y_true = []
    for inputs,label in Testloader:
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        model.eval()
        logits = model(inputs)
        pred = logits.max(dim=1, keepdim=True)[1]
        y_pred.extend(pred.tolist())  
        y_true.extend(label.tolist())
        counter += 1
        if counter % 10 == 0:
            print(f'[{counter}/{len(Testloader)}] ', end='\r')
    y_pred = np.array(y_pred)
    y_true = np.array(y_true)
    metrics = {
            "session_acc" : accuracy_score(y_true, y_pred),
            "f1" : f1_score(y_true, y_pred),
            "recall" : recall_score(y_true, y_pred),
            "precision" : precision_score(y_true, y_pred)}
    print([(k, round(v, 4))for k,v in metrics.items()])


In [23]:
node_dict={}
node_dict['C0'] = 0
for y in range(1,55):
    node_dict['C'+str(y)] = y

In [24]:
# encode_dict = np.load('Thunderbird/com_encode_dict256tb.npy',allow_pickle=True).item()

In [25]:
encode_dict["C0"] = np.zeros((254, 256))

In [26]:
encode_dict['C15'].shape

(4, 256)

In [27]:
# for inputs,label in Trainloader:
#     for g in inputs:
#         g = g.to(device) 
#         for ntype in g.ntypes:
#             # if ntype == 'C15':
#                 # print(g.number_of_nodes(ntype))
#             # print(ntype)
#             # print(encode_dict[ntype])
#             g.nodes[ntype].data["inp"] = torch.tensor(encode_dict[ntype][:g.number_of_nodes(ntype)]).float().cuda() 

NameError: name 'device' is not defined

In [28]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
n_epoch=10
n_hid=256
n_inp=256
clip=1.0
max_lr=1e-3

In [29]:
# del model

In [30]:
model = HGT(
    node_dict,
    encode_dict,
    n_inp=n_inp,
    n_hid=n_hid,
    n_out=2,
    n_layers=1,
    n_heads=1,
    use_norm=True,
).to(device)
optimizer = torch.optim.AdamW(model.parameters())
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer, total_steps=n_epoch, max_lr=max_lr
)

In [31]:
train_step = torch.tensor(0)
start = time.time()
for epoch in range(1,n_epoch + 1):
    train_step+=1
    # Train cycle
    train(model,train_step)
    test(model)

F:\anaconda\envs\wppenv\lib\site-packages\torch\optim\lr_scheduler.py:163: UserWarning: The epoch parameter in `scheduler.step()` was not necessary and is being deprecated where possible. Please use `scheduler.step()` to step the scheduler. During the deprecation, if epoch is different from None, the closed form is used instead of the new chainable form, where available. Please open an issue if you are unable to replicate your use case: https://github.com/pytorch/pytorch/issues/new/choose.
  warnings.warn(EPOCH_DEPRECATION_WARNING, UserWarning)


 21s] Epoch 1 [10/250] loss=0.03212704006582499
 14s] Epoch 1 [20/250] loss=0.019411973752721613
 17s] Epoch 1 [30/250] loss=0.01636927907044684
 21s] Epoch 1 [40/250] loss=0.016717225874799853
 20s] Epoch 1 [50/250] loss=0.015671189247132317
 7s] Epoch 1 [60/250] loss=0.01900414923552505
 53s] Epoch 1 [70/250] loss=0.017489835723625984
 36s] Epoch 1 [80/250] loss=0.018346639480742583
 20s] Epoch 1 [90/250] loss=0.018223026035000013
 7s] Epoch 1 [100/250] loss=0.016887848436374555
 57s] Epoch 1 [110/250] loss=0.01579719021990312
 48s] Epoch 1 [120/250] loss=0.01582693995917457
 38s] Epoch 1 [130/250] loss=0.015112941152838733
 29s] Epoch 1 [140/250] loss=0.014518844697517902
 18s] Epoch 1 [150/250] loss=0.013677959850072995
 13s] Epoch 1 [160/250] loss=0.013457053134057331
 9s] Epoch 1 [170/250] loss=0.01294840645070607
 5s] Epoch 1 [180/250] loss=0.012400872597030004
 1s] Epoch 1 [190/250] loss=0.012026954422678256
 59s] Epoch 1 [200/250] loss=0.011581555181122345
 55s] Epoch 1 [210/2

F:\anaconda\envs\wppenv\lib\site-packages\torch\optim\lr_scheduler.py:163: UserWarning: The epoch parameter in `scheduler.step()` was not necessary and is being deprecated where possible. Please use `scheduler.step()` to step the scheduler. During the deprecation, if epoch is different from None, the closed form is used instead of the new chainable form, where available. Please open an issue if you are unable to replicate your use case: https://github.com/pytorch/pytorch/issues/new/choose.
  warnings.warn(EPOCH_DEPRECATION_WARNING, UserWarning)


KeyboardInterrupt: 

In [ ]:
test(model)

In [ ]:
len(Testloader)

In [ ]:
len(Trainloader)

In [ ]:
for x,y in Testloader:
    sum = 0
    for i in x[0].ntypes:
        sum =sum+x[0].number_of_nodes(i)
    print(sum)
    break